# Capítulo 3 · Actividad: análisis de datos con NumPy y pandas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_03/01_actividad_numpy_pandas.ipynb)

En esta actividad haces de **analista de datos**: vas a responder preguntas sobre datos reales usando solo **NumPy** y **pandas**, sin gráficas. Lo importante no es solo que el código funcione, sino que **sepas interpretar los resultados**.

### Antes de empezar

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia. Si editas el original tendrás conflictos al sincronizar tu fork.
2. Si todavía no tienes los datos, descárgalos desde la raíz del repo:
   ```bash
   uv run python scripts/download_data.py
   ```
3. Ten a mano las notebooks de clase: `notebooks/lectures/chapter_03/01_numpy.ipynb` y `02_pandas.ipynb`.

### Cómo trabajar

- Cada ejercicio tiene una celda `# Tu código aquí`. Puedes añadir todas las celdas que necesites.
- En las celdas ✍️ responde con tus palabras y **apoya tus respuestas con números** de tus resultados.
- Las 💡 **pistas** te recuerdan funciones. Si una función no salió en clase, la pista explica cómo usarla.
- Al terminar, ejecuta **Kernel → Restart & Run All** y comprueba que todo funciona de arriba abajo.

| Parte | Datos | Herramienta |
|---|---|---|
| 1. La cafetería | ventas de un mes (generadas en la notebook) | NumPy |
| 2. Las propinas | `data/tips.csv` | pandas (+ un poco de NumPy) |
| Reto opcional | `data/titanic.csv` | pandas |

In [57]:
from pathlib import Path

import numpy as np
import pandas as pd

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"
print("Carpeta de datos:", DATA_DIR)

Carpeta de datos: /home/tenientez/Documentos/Python/SamsungCurso/data


---
## Parte 1 · La cafetería ☕ (NumPy)

Una cafetería vende 5 productos y ha registrado cuántas unidades vendió de cada uno durante **4 semanas** (28 días, de lunes a domingo). La dueña quiere saber cuánto gana, qué días van mejor y qué producto le conviene más.

Ejecuta la siguiente celda para generar los datos. **No hace falta que la modifiques ni que entiendas cada línea.**

In [58]:
# Datos de la cafetería (no hace falta modificar esta celda)
rng = np.random.default_rng(42)

productos = np.array(["café", "latte", "té", "croissant", "sándwich"])
precios = np.array([1.5, 2.8, 1.2, 2.0, 4.5])       # precio de cada producto, en $

# Unidades vendidas cada día: los fines de semana se vende un 50% más
dia_semana = np.tile(np.arange(7), 4)                # 0 = lunes, ..., 6 = domingo
factor = np.where(dia_semana >= 5, 1.5, 1.0)
promedio = np.outer(factor, [30, 22, 12, 18, 9])     # ventas esperadas por día y producto
ventas_planas = rng.poisson(promedio).ravel()         # ¡todo en una sola fila!

print(ventas_planas[:10], "...")
print("Número de valores:", ventas_planas.size)

[35 28  7 16 13 29 16 10 18 11] ...
Número de valores: 140


### Ejercicio 1 · Dar forma a los datos

`ventas_planas` tiene 140 números seguidos: primero las ventas de los 5 productos del día 1, después las del día 2, y así sucesivamente.

a) Crea una matriz `ventas` con forma `(28, 5)`: una **fila por día** y una **columna por producto** (en el mismo orden que `productos`).

b) Imprime su `shape`, `ndim`, `size` y `dtype`.

c) ¿Cuántos cafés se vendieron el primer día? Comprueba que coincide con `ventas_planas[0]`.

> 💡 **Pista:** `reshape`.

In [59]:
#A) Crea una matriz 'ventas' com forma (28,5)
ventas = ventas_planas.reshape(28,5)
print(ventas)

#B) Datos
print('shape:', ventas.shape)
print('ndim:', ventas.ndim)
print('size:', ventas.size)
print('dtype:', ventas.dtype)

#C) Venta de cafes el primer dia
print('La ventas de cafe el primer dia fue:', ventas[0,0], 'cafes')
print(ventas_planas[0])

[[35 28  7 16 13]
 [29 16 10 18 11]
 [33 17 15 17 10]
 [29 21  9 15  8]
 [34 17  8 17  8]
 [49 29 25 22 14]
 [53 37 20 29 12]
 [30 16 13 26  7]
 [30 23  9 18  6]
 [26 25  9 19  9]
 [38 22 15 16  8]
 [37 19  8 14 10]
 [57 29 20 20 20]
 [44 39 14 29 15]
 [28 24 11 14  7]
 [39 19  7  9  8]
 [24 18 11 16 10]
 [35 22 16 16  7]
 [28 30 10 23 10]
 [50 44 28 28 11]
 [34 27 19 27 17]
 [22 14 11 20  9]
 [34 31 12 27  7]
 [26 21 17 23  7]
 [35 30  3  9 10]
 [23 26 16 21  8]
 [38 32 20 21 11]
 [49 45 18 27  9]]
shape: (28, 5)
ndim: 2
size: 140
dtype: int64
La ventas de cafe el primer dia fue: 35 cafes
35


### Ejercicio 2 · Indexación

Obtén, usando índices y *slicing* sobre `ventas`:

a) Las ventas de todos los productos el **primer día**.

b) Las ventas de **latte** (columna 1) durante todo el mes.

c) Las ventas de la **última semana** (los últimos 7 días). Comprueba que su forma es `(7, 5)`.

d) Las ventas de **croissant y sándwich** (las dos últimas columnas) durante la **primera semana**.

> 💡 **Pista:** `M[filas, columnas]`. Recuerda que los índices negativos cuentan desde el final.

In [60]:
#A) Ventas productos primer dia
ventas_primer_dia = ventas[0].sum()
print('Ventas el primer dia totales:', ventas_primer_dia)

#B) Ventas de latte durante todo el mes
latte_mes = ventas[:,1].sum()
print('Ventas de latte duante todo el mes:', latte_mes)

#C) Ventas ultima semana de y su forma (7,5)
ventas_ultima_semana = ventas[-7:]
print('Matriz ventas ultima semana', ventas_ultima_semana)
print('Dimension ventas ultimas sememana:', ventas_ultima_semana.shape)

#D) Ventas croissant y sandwich durante la primer semana
croissant_sandwich = ventas[:7,-2:] # los dos puntos influyen i quieres hacer para atras o para adelante, 
#pj piden primera semana es de 0 a 6, entonces tomas de 6 y para atras, es tu limite. y con -2,tomas de la penultima para adelante.
print('Ventas croissant y sandwich durante la primer semana', croissant_sandwich)
print('Ventas croissant y sandwich durante la primer semana totales:', croissant_sandwich.sum())

Ventas el primer dia totales: 99
Ventas de latte duante todo el mes: 721
Matriz ventas ultima semana [[22 14 11 20  9]
 [34 31 12 27  7]
 [26 21 17 23  7]
 [35 30  3  9 10]
 [23 26 16 21  8]
 [38 32 20 21 11]
 [49 45 18 27  9]]
Dimension ventas ultimas sememana: (7, 5)
Ventas croissant y sandwich durante la primer semana [[16 13]
 [18 11]
 [17 10]
 [15  8]
 [17  8]
 [22 14]
 [29 12]]
Ventas croissant y sandwich durante la primer semana totales: 210


### Ejercicio 3 · Ingresos

a) Calcula la matriz `ingresos` (forma `(28, 5)`): lo que se ingresó por cada producto cada día.

b) ¿Cuánto ingresó la cafetería en todo el mes?

c) Crea `ingreso_diario`, un array con el ingreso total de **cada día** (forma `(28,)`).

d) ¿Cuál fue el ingreso diario medio, el máximo y el mínimo?

> 💡 **Pistas:**
> - `ventas * precios` multiplica **cada fila** de `ventas` por el vector `precios`, elemento a elemento: la columna del café por 1.5, la del latte por 2.8… (este mecanismo se llama *broadcasting*).
> - `M.sum()` suma todo. `M.sum(axis=1)` suma **cada fila** y devuelve un resultado por fila; `M.sum(axis=0)` suma **cada columna**.

In [61]:
# A) Matriz ingresos de cada dia forma (28, 5)
ingresos = ventas * precios
print(ingresos)
print(ingresos.shape)

#B) ingreso cafeteria todo el mes
ingreso_total = ingresos.sum()
print('La cafeteria ingreso', ingreso_total, 'pesos')

#C) Ingreso diario
ingreso_diario = ingresos.sum(axis=1)
print(ingreso_diario)
print(ingreso_diario.shape)
#Hacemos que sea una columna y no un array
#print(ingreso_diario.reshape(28,1))

#D) Ingreso diario medio, maximo y minimo
print('El ingreso diario medio es:', ingreso_diario.mean().round())
print('El ingreso diario maximo es:', ingreso_diario.max().round())
print('El ingreso diario minimo es:', ingreso_diario.min().round())


[[ 52.5  78.4   8.4  32.   58.5]
 [ 43.5  44.8  12.   36.   49.5]
 [ 49.5  47.6  18.   34.   45. ]
 [ 43.5  58.8  10.8  30.   36. ]
 [ 51.   47.6   9.6  34.   36. ]
 [ 73.5  81.2  30.   44.   63. ]
 [ 79.5 103.6  24.   58.   54. ]
 [ 45.   44.8  15.6  52.   31.5]
 [ 45.   64.4  10.8  36.   27. ]
 [ 39.   70.   10.8  38.   40.5]
 [ 57.   61.6  18.   32.   36. ]
 [ 55.5  53.2   9.6  28.   45. ]
 [ 85.5  81.2  24.   40.   90. ]
 [ 66.  109.2  16.8  58.   67.5]
 [ 42.   67.2  13.2  28.   31.5]
 [ 58.5  53.2   8.4  18.   36. ]
 [ 36.   50.4  13.2  32.   45. ]
 [ 52.5  61.6  19.2  32.   31.5]
 [ 42.   84.   12.   46.   45. ]
 [ 75.  123.2  33.6  56.   49.5]
 [ 51.   75.6  22.8  54.   76.5]
 [ 33.   39.2  13.2  40.   40.5]
 [ 51.   86.8  14.4  54.   31.5]
 [ 39.   58.8  20.4  46.   31.5]
 [ 52.5  84.    3.6  18.   45. ]
 [ 34.5  72.8  19.2  42.   36. ]
 [ 57.   89.6  24.   42.   49.5]
 [ 73.5 126.   21.6  54.   40.5]]
(28, 5)
La cafeteria ingreso 6342.5 pesos
[229.8 185.8 194.1 179.1 178.2 29

### Ejercicio 4 · Días buenos y días flojos

Usa este array con el número de cada día (del 1 al 28):
```python
dias = np.arange(1, 29)
```

a) Crea una máscara con los días cuyo ingreso supera el **ingreso diario medio** y muestra **qué días** fueron.

b) ¿**Cuántos** días fueron?

c) Crea un array que diga `"bueno"` o `"flojo"` para cada día, según si supera o no la media.

d) Calcula `dias[mascara] % 7` (el resto de dividir entre 7). El día 1 fue lunes, así que un resto de **6** es sábado y un resto de **0** es domingo. ¿Qué tienen en común los días buenos?

> 💡 **Pistas:** `.sum()` sobre una máscara cuenta los `True`, porque `True` vale 1 y `False` vale 0. Para c), `np.where`.

In [62]:
dias = np.arange(1,29)
print(dias)

#A) Crea una mascara con los dias cuyo ingreso supera al ingreso medio diario

ingreso_mayor_medio_diario = ingreso_diario > ingreso_diario.mean() #inicializamos la mascara
print('Macara con resultados booleanos: ', ingreso_mayor_medio_diario)
print(dias[ingreso_mayor_medio_diario]) #imprimimos los dias que tuvieron mas del ingreso medio diario

#Al tener ambos la misma longitud (28 elementos), la máscara actúa como un colador sobre dias, 
# dejando pasar únicamente el número del día donde hubo un True.

#B) Cuantos dias fueron?

print('Los dias donde hubo ingreso diario por encima de la media en total son: ', ingreso_mayor_medio_diario.sum())

#C) Array que diga dia flojo o dia bueno, segun si supera la media

calidad_dia = np.where(ingreso_mayor_medio_diario, 'Bueno', 'Flojo')
print(calidad_dia)

#D) Calcular que tienen en comun los dias buenos y los dias flojos

residuo = dias[ingreso_mayor_medio_diario] % 7 #Sabado es 6 y Domingo 0
print(residuo)
#Los dias buenos son el fin de semana:)

[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28]
Macara con resultados booleanos:  [ True False False False False  True  True False False False False False
  True  True False False False False  True  True  True False  True False
 False False  True  True]
[ 1  6  7 13 14 19 20 21 23 27 28]
Los dias donde hubo ingreso diario por encima de la media en total son:  11
['Bueno' 'Flojo' 'Flojo' 'Flojo' 'Flojo' 'Bueno' 'Bueno' 'Flojo' 'Flojo'
 'Flojo' 'Flojo' 'Flojo' 'Bueno' 'Bueno' 'Flojo' 'Flojo' 'Flojo' 'Flojo'
 'Bueno' 'Bueno' 'Bueno' 'Flojo' 'Bueno' 'Flojo' 'Flojo' 'Flojo' 'Bueno'
 'Bueno']
[1 6 0 6 0 5 6 0 2 6 0]


✍️ **Tu respuesta:** 

### Ejercicio 5 · Análisis por semanas

a) Reorganiza `ingreso_diario` en una matriz `por_semana` de forma `(4, 7)`: una fila por semana y una columna por día (lunes a domingo).

b) ¿Cuánto se ingresó **cada semana**?

c) ¿Cuál es el ingreso medio de **cada día de la semana**? ¿Qué día se vende más? Usa:
```python
nombres_dias = np.array(["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"])
```

d) Compara el ingreso medio de un día de **fin de semana** con el de un día **entre semana**. ¿Cuántas veces más se ingresa en fin de semana?

> 💡 **Pistas:** `axis=1` da un resultado por fila (por semana) y `axis=0` uno por columna (por día de la semana). `arr.argmax()` devuelve la **posición** del valor máximo; úsala como índice en `nombres_dias`.

In [63]:
#A) Reorganizar en matriz (4,7)
print(ingreso_diario)
por_semana = ingreso_diario.reshape(4,7)
print(por_semana) #reorganizamos el array y lo sacamos again

#B) Cuánto se ingreso por semana?

ingreso_por_semana = por_semana.sum(axis=1)
print('El ingreso por semana es: ', ingreso_por_semana)

#C) Cual es el ingreso medio po dia de la semana, que dia se vende mas?

nombres_dias = np.array(['Lunes', 'Martes', 'Miercoles', 'JUeves', 'Viernes', 'Sabado', 'Domingo'])
print(nombres_dias)
ingreso_medio_por_semana = por_semana.mean(axis=0)
print('Ingreso medio por dia de la semana: ', ingreso_medio_por_semana)
print('El dia que mas se vende es: ', nombres_dias[np.argmax(ingreso_medio_por_semana)]) 
#argmax, en ves de darte el maximo, te da la posicion donde esta el maximo

#D) Comparar ingreso medio fin de semana vs la semana

ingreso_medio_entresemana = por_semana[0:,0:5].mean()
print('EL ingreso medio en la semana es: ', ingreso_medio_entresemana)
ingreso_medio_finsemana = por_semana[0:,-2:].mean()
print('El ingreso medio el fin de semana es: ', ingreso_medio_finsemana)
#El ingreso en fin de semana es casi el doble que entre semana
print('El fin de semana se gana',(ingreso_medio_finsemana/ingreso_medio_entresemana),'mas que entre semana')


[229.8 185.8 194.1 179.1 178.2 291.7 319.1 188.9 183.2 198.3 204.6 191.3
 320.7 317.5 181.9 174.1 176.6 196.8 229.  337.3 279.9 165.9 237.7 195.7
 203.1 204.5 262.1 315.6]
[[229.8 185.8 194.1 179.1 178.2 291.7 319.1]
 [188.9 183.2 198.3 204.6 191.3 320.7 317.5]
 [181.9 174.1 176.6 196.8 229.  337.3 279.9]
 [165.9 237.7 195.7 203.1 204.5 262.1 315.6]]
El ingreso por semana es:  [1577.8 1604.5 1575.6 1584.6]
['Lunes' 'Martes' 'Miercoles' 'JUeves' 'Viernes' 'Sabado' 'Domingo']
Ingreso medio por dia de la semana:  [191.625 195.2   191.175 195.9   200.75  302.95  308.025]
El dia que mas se vende es:  Domingo
EL ingreso medio en la semana es:  194.93
El ingreso medio el fin de semana es:  305.4875
El fin de semana se gana 1.5671651362027395 mas que entre semana


### Ejercicio 6 · El producto estrella

a) Calcula el ingreso total de **cada producto** en el mes.

b) ¿Qué **porcentaje** del ingreso total aporta cada producto? Redondea a 1 decimal.

c) ¿Qué producto genera **más dinero**? ¿Y cuál vende **más unidades**?

> 💡 **Pista:** `axis=0` y `argmax`, igual que en el ejercicio anterior.

In [ ]:
# A) Ingreso total de cada producto en un solo array de 5 elementos
ingreso_por_producto = ingresos.sum(axis=0)
print('Ingeso por producto: ', ingreso_por_producto)

# B) Porcentaje de cada producto (operación vectorizada directa)
porcentaje_por_producto = (ingreso_por_producto / ingreso_total * 100).round(1)
print('Porcentaje por prodcuto ', porcentaje_por_producto)

#C) Que producto genera mas dinero y cual vende mas unidades
producto_mas_ingresos = productos[np.argmax(ingreso_por_producto)]
print('El producto que mas ingresos dio es: ', producto_mas_ingresos)

total_por_producto = ventas.sum(axis=0)
print('El total de unidades por producto es: ', total_por_producto)
producto_mas_unidades = productos[np.argmax(total_por_producto)]
print('El producto que mas unidades vendio es: ', producto_mas_unidades)

Ingeso por producto:  [1483.5 2018.8  457.2 1114.  1269. ]
Porcentaje por prodcuto  [23.4 31.8  7.2 17.6 20. ]
El producto que mas ingresos dio es:  latte
El total de unidades por producto es:  [989 721 381 557 282]
El producto que mas unidades vendio es:  café


'\nidx_mas_vendido = np.argmax(ventas)\nproducto_mas_unidades = productos[np.argmax(idx_mas_vendido)]\n'

Se hace lo mismo arriba y abajo, pero arriba esta mejor escirto

In [ ]:
#A) Ingreso total de cada producto en el mes
ventas_cafe_total_mes = ingresos[0:, 0].sum()
print('El ingreso total de cafe en el mes es: ', ventas_cafe_total_mes)
ventas_latte_total_mes = ingresos[0:, 1].sum()
print('El ingreso total de latte en el mes es: ', ventas_latte_total_mes)
ventas_te_total_mes = ingresos[0:, 2].sum()
print('El ingreso total de te en el mes es: ', ventas_te_total_mes)
ventas_croissant_total_mes = ingresos[0:, 3].sum()
print('El ingreso total de crisstant en el mes es: ', ventas_croissant_total_mes)
ventas_sandwich_total_mes = ingresos[0:, 4].sum()
print('El ingreso total de sandwich en el mes es: ', ventas_sandwich_total_mes)

#B) Que porcentaje del ingreso total aporta cada producto
def porcentaje_total_mes(a, ingreso_total):
    """Calcula el porcentaje de lo que aporto el producto en el mes"""
    porcentaje = ((a*100)/ingreso_total).round(1)
    return porcentaje

print('El ingreso total en el mes es: ', ingreso_total)
print('El cafe aporto el',porcentaje_total_mes(ventas_cafe_total_mes, ingreso_total),'del ingreso total')
print('El latte aporto el',porcentaje_total_mes(ventas_latte_total_mes, ingreso_total),'del ingreso total')
print('El te aporto el',porcentaje_total_mes(ventas_te_total_mes, ingreso_total),'del ingreso total')
print('El croissant aporto el',porcentaje_total_mes(ventas_croissant_total_mes, ingreso_total),'del ingreso total')
print('El sandwich aporto el',porcentaje_total_mes(ventas_sandwich_total_mes, ingreso_total),'del ingreso total')

#C) Que producto genera mas dinero y cual vende mas unidades
#Viendo los resultad, el latte es el que genera mas dinero y el que vende mas unidades es el 


El ingreso total de cafe en el mes es:  1483.5
El ingreso total de latte en el mes es:  2018.7999999999997
El ingreso total de te en el mes es:  457.20000000000005
El ingreso total de crisstant en el mes es:  1114.0
El ingreso total de sandwich en el mes es:  1269.0
El ingreso total en el mes es:  6342.5
El cafe aporto el 23.4 del ingreso total
El latte aporto el 31.8 del ingreso total
El te aporto el 7.2 del ingreso total
El croissant aporto el 17.6 del ingreso total
El sandwich aporto el 20.0 del ingreso total


✍️ **Pregunta:** ¿por qué el producto que más se vende no es el que más dinero deja? Si fueras la dueña, ¿qué harías con esta información?

✍️ **Tu respuesta:** 

---
## Parte 2 · Las propinas 💵 (pandas)

Un camarero anotó, durante varias semanas, cada mesa que atendió: cuánto pagaron, cuánta propina dejaron y algunos datos de la mesa. Queremos responder una pregunta: **¿de qué depende la propina?**

El archivo es `data/tips.csv` y tiene estas columnas:

| Columna | Significado |
|---|---|
| `total_bill` | importe de la cuenta, en $ |
| `tip` | propina, en $ |
| `sex` | sexo de quien pagó (`Female` / `Male`) |
| `smoker` | si en la mesa había fumadores (`Yes` / `No`) |
| `day` | día de la semana (`Thur`, `Fri`, `Sat`, `Sun`) |
| `time` | `Lunch` (comida) o `Dinner` (cena) |
| `size` | número de personas en la mesa |

### Ejercicio 7 · Primer vistazo

a) Carga el archivo en un DataFrame llamado `propinas`.

b) Explóralo con `shape`, `head`, `info` y `describe`.

c) ¿Cuántas mesas hay de cada día? ¿Y de cada momento (comida/cena)?

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

✍️ **Pregunta:** ¿cuántas mesas hay? ¿Falta algún dato? ¿Cuál fue la cuenta más alta y cuál la mesa más grande?

✍️ **Tu respuesta:** 

### Ejercicio 8 · Traducir los datos

Para trabajar más cómodos, vamos a pasar todo a español.

a) Renombra las columnas: `total_bill` → `cuenta`, `tip` → `propina`, `sex` → `sexo`, `smoker` → `fumador`, `day` → `dia`, `time` → `momento`, `size` → `personas`.

b) Traduce los valores:
- `dia`: `Thur` → `Jueves`, `Fri` → `Viernes`, `Sat` → `Sábado`, `Sun` → `Domingo`
- `momento`: `Lunch` → `Comida`, `Dinner` → `Cena`
- `sexo`: `Female` → `Mujer`, `Male` → `Hombre`
- `fumador`: `Yes` → `sí`, `No` → `no`

Comprueba el resultado con `head()`.

> 💡 **Pista:** `rename(columns={...})` y `.map({...})`.

In [ ]:
# Tu código aquí

### Ejercicio 9 · Nuevas columnas

Una propina de 5 $ no significa lo mismo en una cuenta de 15 $ que en una de 50 $. Por eso vamos a crear variables más útiles:

a) `pct_propina`: la propina como **porcentaje** de la cuenta.

b) `cuenta_por_persona`: lo que pagó cada persona de la mesa.

c) `tamaño_mesa`: agrupa `personas` en `"1-2"`, `"3-4"` y `"5-6"`.

d) `generosa`: `"sí"` si la propina es del 20% o más, `"no"` en caso contrario.

> 💡 **Pistas:** operaciones entre columnas, `pd.cut` (con `bins=[0, 2, 4, 6]`) y `np.where`.

In [ ]:
# Tu código aquí

### Ejercicio 10 · Filtrar y ordenar

a) ¿Cuántas mesas dejaron una propina **generosa** (20% o más)? ¿Qué **proporción** del total representan?

b) ¿Cuántas **cenas de fin de semana** (sábado o domingo) fueron de **4 personas o más**? ¿Cuál fue su porcentaje de propina medio?

c) Muestra las **5 mesas** con mayor `pct_propina`, con las columnas `cuenta`, `propina`, `pct_propina`, `personas` y `dia`.

d) Usa `query` para obtener las mesas con **fumadores** y una cuenta **entre 10 y 20 $**. ¿Cuántas son?

> 💡 **Pistas:** filtros con `&`, `isin` y `sort_values`. Dentro de `query` el texto va entre comillas simples: `"fumador == 'sí'"`.

In [ ]:
# Tu código aquí

✍️ **Pregunta:** fíjate en la primera mesa del apartado c). ¿Te parece un caso normal? ¿Cómo crees que afecta a la **media** del porcentaje de propina?

✍️ **Tu respuesta:** 

### Ejercicio 11 · Agrupar y resumir

a) Para cada **día**, calcula el número de mesas, la **media** y la **mediana** de `pct_propina`.

b) Para cada **momento** (comida/cena), crea un resumen con columnas con nombre: `n_mesas`, `cuenta_media`, `propina_media` y `pct_medio`.

c) Para cada número de **personas**, calcula el número de mesas y el `pct_propina` medio.

d) Crea una tabla dinámica con el `pct_propina` medio por **sexo** (filas) y **fumador** (columnas).

e) Quédate solo con las mesas del **domingo** y calcula la media y la mediana de `pct_propina` **con** y **sin** la mesa del 71% que encontraste en el ejercicio 10.

> 💡 **Pistas:** `groupby(...).agg([...])`, `agg(nombre=("columna", "función"))` y `pd.pivot_table`. Redondea con `.round(1)` para que se lea mejor.

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

✍️ **Preguntas:**
1. En a), ¿en qué días la media es mayor que la mediana? Según e), ¿cuál de las dos medidas se ve más afectada por una mesa extrema? ¿Cuál usarías para describir la propina "típica"?
2. En c), ¿las mesas más grandes dejan más o menos porcentaje de propina? ¿Te fiarías de los resultados para 1, 5 o 6 personas? ¿Por qué?

✍️ **Tu respuesta:** 

### Ejercicio 12 · Cuidado con las comparaciones

Haz una tabla cruzada que cuente cuántas mesas hay de cada **día** (filas) en cada **momento** (columnas).

> 💡 **Pista:** `pd.crosstab`.

In [ ]:
# Tu código aquí

✍️ **Pregunta:** si comparas las propinas de la comida con las de la cena, ¿estás comparando solo el momento del día, o también otra cosa? ¿Por qué es importante tenerlo en cuenta antes de sacar conclusiones?

✍️ **Tu respuesta:** 

### Ejercicio 13 · Combinar tablas

Tenemos otra tabla con información de cada día:
```python
info_dias = pd.DataFrame({
    "dia": ["Jueves", "Viernes", "Sábado", "Domingo"],
    "orden": [4, 5, 6, 7],
    "fin_de_semana": [False, False, True, True],
})
```

a) Añade esta información a `propinas` con `merge`. Comprueba que el número de filas **no cambia**.

b) Compara la **cuenta media** y el **`pct_propina` medio** entre fin de semana y entre semana.

c) Calcula cuánto se facturó en total **cada día**, ordenado cronológicamente (de jueves a domingo).

> 💡 **Pista:** para c), agrupa por `["orden", "dia"]`: `groupby` ordena los grupos por la primera columna.

In [ ]:
# Tu código aquí

### Ejercicio 14 · De pandas a NumPy

Los modelos de Machine Learning trabajan con arrays de NumPy, así que es habitual pasar de un DataFrame a un array.

a) Crea `X` con las columnas `cuenta`, `propina` y `personas` convertidas a array de NumPy. ¿Qué forma tiene?

b) **Solo con NumPy**, calcula el porcentaje de propina de cada mesa a partir de `X`. Comprueba que su media coincide con la de `propinas["pct_propina"]`.

c) **Solo con NumPy**, ¿cuántas mesas de **2 personas** dejaron **más de 4 $** de propina?

> 💡 **Pistas:** `.to_numpy()`. En `X`, la columna 0 es la cuenta, la 1 la propina y la 2 las personas.

In [ ]:
# Tu código aquí

### ✍️ Conclusiones de la Parte 2

Imagina que el camarero te pregunta: *"¿qué tengo que hacer para conseguir mejores propinas?"*. Escribe:

1. **Tres hallazgos** sobre las propinas, cada uno apoyado en un número concreto de tus resultados.
2. **Una limitación** de estos datos que haga que debas tener cuidado con tus conclusiones.

✍️ **Tu respuesta:** 

---
## Reto opcional · El Titanic 🚢

En 1912 el Titanic se hundió en su viaje inaugural. El archivo `data/titanic.csv` tiene información de 891 pasajeros. Las columnas que necesitas:

| Columna | Significado |
|---|---|
| `survived` | 1 si sobrevivió, 0 si no |
| `sex` | `male` / `female` |
| `age` | edad (¡algunas faltan!) |
| `fare` | precio del billete |
| `class` | clase del billete: `First`, `Second`, `Third` |

**Pregunta:** ¿se cumplió el famoso *"las mujeres y los niños primero"*? ¿Importaba la clase del billete?

a) Carga los datos y usa `info()` para ver qué columnas tienen valores faltantes.

b) ¿Qué proporción de pasajeros sobrevivió?

c) Calcula la tasa de supervivencia por **sexo**, por **clase** y en una tabla dinámica **sexo × clase**.

d) Agrupa las edades en `"niño"` (0–12), `"adolescente"` (13–18), `"adulto"` (19–60) y `"mayor"` (más de 60), y calcula la tasa de supervivencia y el número de pasajeros de cada grupo.

e) ¿Cuánto costaba de media el billete en cada clase?

> 💡 **Pistas:**
> - La **media** de una columna de 0 y 1 es la **proporción** de unos: `titanic["survived"].mean()` es la tasa de supervivencia.
> - Para d), `pd.cut` con `bins=[0, 12, 18, 60, 100]`. Los pasajeros sin edad (`NaN`) quedan fuera de los grupos automáticamente.

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

✍️ **Pregunta:** ¿se cumplió *"las mujeres y los niños primero"*? ¿Y la clase, importaba? Responde con números.

✍️ **Tu respuesta:** 